# Embeddings

This lab is designed to help you solidify your understanding of embeddings by applying them to tasks like semantic similarity, clustering, and building a semantic search system.

### Tasks:
- Task 1: Semantic Similarity Comparison
- Task 2: Document Clustering
- Task 3: Enhance the Semantic Search System


## Task 1: Semantic Similarity Comparison
### Objective:
Compare semantic similarity between pairs of sentences using cosine similarity and embeddings.

### Steps:
1. Load a pre-trained Sentence Transformer model.
2. Encode the sentence pairs.
3. Compute cosine similarity for each pair.

### Dataset:
- "A dog is playing in the park." vs. "A dog is running in a field."
- "I love pizza." vs. "I enjoy ice cream."
- "What is AI?" vs. "How does a computer learn?"


In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Sentence pairs
sentence_pairs = [
    ("A dog is playing in the park.", "A dog is running in a field."),
    ("I love pizza.", "I enjoy ice cream."),
    ("What is AI?", "How does a computer learn?")
]

# Compute similarities
similarities = []
for i, (s1, s2) in enumerate(sentence_pairs, 1):
    e1, e2 = model.encode([s1, s2], normalize_embeddings=True)
    sim = cosine_similarity([e1], [e2])[0][0]
    similarities.append(sim)
    print(f"Pair {i} similarity: {sim:.4f}")

Pair 1 similarity: 0.5220
Pair 2 similarity: 0.5281
Pair 3 similarity: 0.3194


### Questions:
- Which sentence pairs are the most semantically similar? Why?
- Can you think of cases where cosine similarity might fail to capture true semantic meaning?


### Answer

**Most similar (by your scores):**
- **Pair 2 (0.5281)** — *"I love pizza."* vs *"I enjoy ice cream."*  
  Both express positive preference about food; strong semantic overlap.  
- **Pair 1 (0.5220)** — *"A dog is playing…"* vs *"A dog is running…"*.  
  Same subject/action context; near-synonyms, different setting.  
- **Least: Pair 3 (0.3194)** — definition vs mechanism; different intent.  

**When cosine similarity can miss meaning:**
- **Negation/antonyms:** “good” vs “not good.”  
- **Role/order flips:** “Dog bites man” vs “Man bites dog.”  
- **Sarcasm/pragmatics:** “Great job 🙄” (actually negative).  
- **Numbers/details:** “up 5%” vs “down 5%.”  
- **Polysemy:** “bank” (river) vs “bank” (finance).  

## Task 2: Document Clustering
### Objective:
Cluster a set of text documents into similar groups based on their embeddings.

### Steps:
1. Encode the documents using Sentence Transformers.
2. Use KMeans clustering to group the documents.
3. Analyze the clusters for semantic meaning.

In [2]:
from sklearn.cluster import KMeans

# Documents to cluster
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?"
]

# Encode documents

embeddings = model.encode(documents, normalize_embeddings=True)
print(embeddings.shape)

(6, 384)


In [3]:
# Perform KMeans clustering

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(embeddings)

In [4]:
# Print cluster assignments

clusters = {i: [] for i in range(3)}
for doc, lab in zip(documents, labels):
    clusters[lab].append(doc)

for cid, docs in clusters.items():
    print(f"\nCluster {cid}:")
    for d in docs:
        print(" -", d)


Cluster 0:
 - How do I change a flat tire on a car?
 - How do I fix a leaky faucet?

Cluster 1:
 - What is the capital of France?
 - How do I bake a chocolate cake?
 - What is the distance between Earth and Mars?

Cluster 2:
 - What is the best way to learn Python?


### Questions:
- How many clusters make the most sense? Why?
- Examine the documents in each cluster. Are they semantically meaningful? Can you assign a semantic "theme" to each cluster?
- Try this exercise with a larger dataset of your choice

### Answer

**How many clusters make the most sense?**  
- **3 clusters** seem most reasonable:  
  1. DIY / practical tasks.  
  2. Factual knowledge questions.  
  3. Learning/programming.  

**Are the clusters semantically meaningful? Assign themes:**  
- **Cluster 0:** *Change a flat tire / Fix a leaky faucet* → **DIY / Repairs**  
- **Cluster 1:** *Capital of France / Earth–Mars distance* → **General knowledge**  
  *(Baking a cake could also be seen as DIY, though the algorithm grouped it here.)*  
- **Cluster 2:** *Learn Python* → **Programming / Learning**

## Task 3: Semantic Search System
### Objective:
Create a semantic search engine:
A user provides a query and you search the dataset for semantically relevant documents to return. Return the top 5 results.

### Dataset:
- Use the following set of documents:
    - "What is the capital of France?"
    - "How do I bake a chocolate cake?"
    - "What is the distance between Earth and Mars?"
    - "How do I change a flat tire on a car?"
    - "What is the best way to learn Python?"
    - "How do I fix a leaky faucet?"
    - "What are the best travel destinations in Europe?"
    - "How do I set up a local server?"
    - "What is quantum computing?"
    - "How do I build a mobile app?"


In [5]:
import numpy as np

# Documents dataset
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?",
    "What are the best travel destinations in Europe?",
    "How do I set up a local server?",
    "What is quantum computing?",
    "How do I build a mobile app?"
]

# Compute document embeddings

doc_embeddings = model.encode(documents, normalize_embeddings=True)

In [6]:
# Create the search function
# This function should encode the user query and return the top N documents that most resemble it

def semantic_search(query, documents, doc_embeddings, top_n=5):

    # Encode query
    q_emb = model.encode([query], normalize_embeddings=True)[0]

    # Cosine similarity == dot product when embeddings are normalized
    scores = doc_embeddings @ q_emb
    top_idx = np.argsort(-scores)[:top_n]
    return [(documents[i], float(scores[i])) for i in top_idx]

In [7]:
# Test the search function
query = "Explain programming languages."
semantic_search(query, documents, doc_embeddings)

[('What is quantum computing?', 0.4352477788925171),
 ('What is the best way to learn Python?', 0.31878262758255005),
 ('How do I build a mobile app?', 0.11044071614742279),
 ('How do I set up a local server?', 0.09112653136253357),
 ('What are the best travel destinations in Europe?', 0.09064781665802002)]

### Questions:
- What are the top-ranked results for the given queries?
- How can you improve the ranking explanation for users?
- Try this approach with a larger dataset

### Answers

**Top-ranked results (for *“Explain programming languages.”*):**  
1. What is quantum computing?  
2. What is the best way to learn Python?  
3. How do I build a mobile app?  
4. How do I set up a local server?  
5. What are the best travel destinations in Europe?  

**How to improve ranking & explanations:**  
- Use a **cross-encoder** or **hybrid search** (BM25 + embeddings).  
- Highlight matched keywords/snippets for transparency.  
- Expand/rewrite queries to better capture intent.